<a href="https://colab.research.google.com/github/dineshaimldev/Computer-Vision/blob/main/Vision_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [55]:
import torch
import torchvision
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
import torch.utils.data as dataloader
import torch.nn as nn
from torch.nn.modules.transformer import TransformerEncoder

In [56]:
transformation_operation = transforms.Compose([
    transforms.ToTensor()
   ])

In [57]:
train_dataset = torchvision.datasets.MNIST(root='./data',train=True,download=True,transform=transformation_operation)
val_dataset = torchvision.datasets.MNIST(root='./data',train=False,download=True,transform=transformation_operation)


In [58]:
batch_size = 64
num_classes = 10
img_size = 28
num_channels = 1
patch_size = 7
patch_num = int((img_size / patch_size) * (img_size / patch_size))
attention_heads = 8
embed_dim = 64
transformer_blocks = 6
mlp_nodes = 64
learning_rate = 0.001
epoch = 10

In [59]:
train_data = dataloader.DataLoader(train_dataset,batch_size=64,shuffle=True)
val_data = dataloader.DataLoader(val_dataset,batch_size=64,shuffle=False)

In [60]:
class PatchEmbedding(nn.Module):
  def __init__(self):
    super().__init__()
    self.embed = nn.Conv2d(num_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
  def forward(self, x):
    x = self.embed(x)
    x = x.flatten(2).transpose(1, 2)
    return x

In [61]:
class CustomTransformerEncoder(nn.Module):
  def __init__(self):
    super().__init__()
    self.layer_norm1 = nn.LayerNorm(embed_dim)
    self.multi_head_attention = nn.MultiheadAttention(embed_dim, attention_heads, batch_first=True)
    self.layer_norm2 = nn.LayerNorm(embed_dim)
    self.mlp = nn.Sequential(
        nn.Linear(embed_dim, mlp_nodes),
        nn.GELU(),
        nn.Linear(mlp_nodes, embed_dim)
    )
  def forward(self, x):
    residual1 = x
    x = self.layer_norm1(x)
    x = self.multi_head_attention(x, x, x)[0] + residual1
    residual2 = x
    x = self.layer_norm2(x)
    x = self.mlp(x) + residual2
    return x

In [62]:
class MLP_Head(nn.Module):
  def __init__(self):
    super().__init__()
    self.layer_norm1 = nn.LayerNorm(embed_dim)
    self.mlp_head = nn.Sequential(
        nn.Linear(embed_dim, num_classes)
    )
  def forward(self, x):
    x = x[:, 0]
    x = self.layer_norm1(x)
    x = self.mlp_head(x)
    return x

In [63]:
class TransformerVision(nn.Module):
  def __init__(self):
    super().__init__()
    self.patch_embed = PatchEmbedding()
    self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
    self.position_embedding = nn.Parameter(torch.randn(1, patch_num + 1, embed_dim))
    self.transformer_blocks = nn.Sequential(
        *[CustomTransformerEncoder() for _ in range(transformer_blocks)]
    )
    self.mlp_head = MLP_Head()
  def forward(self, x):
    x = self.patch_embed(x)
    B = x.shape[0]
    cls_token = self.cls_token.expand(B, -1, -1)
    x = torch.cat((cls_token, x), dim=1)
    x = x + self.position_embedding
    x = self.transformer_blocks(x)
    x = self.mlp_head(x)
    return x

In [64]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TransformerVision().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

In [65]:
for ep in range(epoch):
  train_acc = 0
  model.train()
  total_images = 0
  correct_images = 0
  total_loss = 0.0
  for batch_idx, (images, labels) in enumerate(train_data):
    images = images.to(device)
    labels = labels.to(device)
    optimizer.zero_grad()
    output = model(images)
    loss = criterion(output, labels)
    loss.backward()
    optimizer.step()
    total_loss += loss.item() * images.size(0)
    predictions = output.argmax(dim=1)
    correct_prediction = (predictions == labels).sum().item()
    correct_images = correct_images + correct_prediction
    total_images = total_images + images.size(0)
    if batch_idx % 100 == 0:
      accuracy = 100.0 * correct_prediction / images.size(0)
      print(f"Epoch {ep+1} | Batch {batch_idx+1:3d} : Loss = {loss.item():.4f}, Accuracy = {accuracy:.2f}%")
  epoch_acc = 100.0 * correct_images / total_images
  epoch_loss = total_loss / total_images
  print(f"Epoch {ep+1} Summary: Total Loss = {epoch_loss:.4f}, Accuracy = {epoch_acc:.2f}%\n")

Epoch 1 | Batch   1 : Loss = 2.3556, Accuracy = 12.50%
Epoch 1 | Batch 101 : Loss = 0.6069, Accuracy = 81.25%
Epoch 1 | Batch 201 : Loss = 0.3086, Accuracy = 92.19%
Epoch 1 | Batch 301 : Loss = 0.4327, Accuracy = 89.06%
Epoch 1 | Batch 401 : Loss = 0.2421, Accuracy = 92.19%
Epoch 1 | Batch 501 : Loss = 0.2227, Accuracy = 95.31%
Epoch 1 | Batch 601 : Loss = 0.1214, Accuracy = 98.44%
Epoch 1 | Batch 701 : Loss = 0.1007, Accuracy = 93.75%
Epoch 1 | Batch 801 : Loss = 0.0794, Accuracy = 98.44%
Epoch 1 | Batch 901 : Loss = 0.2663, Accuracy = 93.75%
Epoch 1 Summary: Total Loss = 0.3549, Accuracy = 88.70%

Epoch 2 | Batch   1 : Loss = 0.1427, Accuracy = 96.88%
Epoch 2 | Batch 101 : Loss = 0.0612, Accuracy = 98.44%
Epoch 2 | Batch 201 : Loss = 0.1495, Accuracy = 96.88%
Epoch 2 | Batch 301 : Loss = 0.0522, Accuracy = 100.00%
Epoch 2 | Batch 401 : Loss = 0.0228, Accuracy = 100.00%
Epoch 2 | Batch 501 : Loss = 0.0343, Accuracy = 98.44%
Epoch 2 | Batch 601 : Loss = 0.0570, Accuracy = 98.44%
Epoch 

In [67]:
model.eval()
val_loss = 0.0
correct_val_images = 0
total_val_images = 0

with torch.no_grad():
  for images, labels in val_data:
    images = images.to(device)
    labels = labels.to(device)
    output = model(images)
    loss = criterion(output, labels)
    val_loss += loss.item() * images.size(0)
    predictions = output.argmax(dim=1)
    correct_prediction = (predictions == labels).sum().item()
    correct_val_images += correct_prediction
    total_val_images += images.size(0)

avg_val_loss = val_loss / total_val_images
val_accuracy = 100.0 * correct_val_images / total_val_images
print(f"Validation Summary: Average Loss = {avg_val_loss:.4f}, Accuracy = {val_accuracy:.2f}%")

Validation Summary: Average Loss = 0.0708, Accuracy = 97.97%


In [66]:
images, labels = next(iter(train_data))
patch_embed = nn.Conv2d(num_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
embedded_image = patch_embed(images)
print("Images shape:", images.shape)
print("After Conv2D projection:", embedded_image.shape)
print("Flattened space:", embedded_image.flatten(2).shape)
print("Transposed (Sequence, Channels):", embedded_image.flatten(2).transpose(1, 2).shape)

Images shape: torch.Size([64, 1, 28, 28])
After Conv2D projection: torch.Size([64, 64, 4, 4])
Flattened space: torch.Size([64, 64, 16])
Transposed (Sequence, Channels): torch.Size([64, 16, 64])
